# Learning & Validation Curves

The [metrics deep-dive](metrics-deep-dive.ipynb) told you *how good* a model is.
This chapter diagnoses *why* — is it **underfitting** (too simple, high bias) or
**overfitting** (too complex, high variance)? — and whether **more data** would
help. Two diagnostic plots answer these:

- a **learning curve** varies the *training-set size*;
- a **validation curve** varies a single *hyperparameter*.

They look similar but ask different questions — keeping them distinct is the
point of this chapter. We reuse `smartcore`'s breast-cancer data throughout, and
let [`model-selection-rs`](https://crates.io/crates/model-selection-rs)'s
`learning_curve` / `validation_curve` do the cross-validated subsetting — wrapping
the `smartcore` models in a small ndarray fit-closure.

In [ ]:
:dep smartcore = { version = "0.3", features = ["datasets"] }
:dep plotters = { version = "0.3", default-features = false, features = ["evcxr", "all_series", "all_elements"] }
:dep ndarray = { version = "0.16" }
:dep model-selection-rs = { version = "0.1.0" }
use smartcore::linalg::basic::matrix::DenseMatrix;
use smartcore::dataset::breast_cancer;
use smartcore::linear::logistic_regression::LogisticRegression;
use plotters::prelude::*;
use ndarray::{Array1, Array2};

// Load breast-cancer as ndarray: an f64 feature matrix + f64 0/1 labels — the
// shapes `model-selection-rs`'s evaluate functions consume. It handles the CV
// subsetting for both curves below, so we no longer split indices by hand.
let (xa, ya): (Array2<f64>, Array1<f64>) = {
    let ds = breast_cancer::load_dataset();
    let xa = Array2::from_shape_vec((ds.num_samples, ds.num_features),
                ds.data.iter().map(|&v| v as f64).collect()).unwrap();
    let ya = Array1::from(ds.target.iter().map(|&v| v as f64).collect::<Vec<f64>>());
    (xa, ya)
};
println!("dataset: {} samples x {} features", xa.nrows(), xa.ncols());

In [ ]:
// Bridge: `model-selection-rs` works on ndarray `Array2<f64>`; `smartcore` models
// want a `DenseMatrix<f32>`. This converts the (sub)matrix the CV machinery hands
// each fit-closure below (row-major, matching ndarray's default layout).
fn to_dm(x: &Array2<f64>) -> DenseMatrix<f32> {
    DenseMatrix::new(x.nrows(), x.ncols(), x.iter().map(|&v| v as f32).collect(), false)
}
println!("to_dm bridge ready");

## Learning curve — does more data help?

Train the model on growing slices of the data; at each size, score it on the
training slice **and** on the held-out folds (averaged over 5-fold CV). Reading
the two curves:

- a large **gap** (train ≫ validation) → **high variance / overfitting** — more
  data or regularization should help;
- both curves **plateau low and together** → **high bias / underfitting** — more
  data won't help; you need a more flexible model or better features.

In [ ]:
use model_selection_rs::splitters::KFold as MsKFold;
use model_selection_rs::evaluate::{learning_curve, TrainSize};
use model_selection_rs::scoring::Accuracy;

// `learning_curve` grows the training subset and averages train/val accuracy over
// the CV folds for us. We wrap smartcore's LogisticRegression in a fit-closure:
// it takes an ndarray (sub)matrix + labels, fits, and returns a predict closure.
let (tr_curve, va_curve): (Vec<(f64, f64)>, Vec<(f64, f64)>) = {
    let fit_lr = |x: &Array2<f64>, y: &Array1<f64>| {
        let yi: Vec<i32> = y.iter().map(|&v| v as i32).collect();
        let model = LogisticRegression::fit(&to_dm(x), &yi, Default::default()).unwrap();
        move |xq: &Array2<f64>| Array1::from(
            model.predict(&to_dm(xq)).unwrap().iter().map(|&v| v as f64).collect::<Vec<f64>>())
    };
    let kf = MsKFold::new(5).unwrap().with_shuffle(42);
    let fracs = [0.1, 0.325, 0.55, 0.775, 1.0].map(TrainSize::Fraction);
    let lc = learning_curve(&kf, &xa, &ya, fit_lr, &Accuracy, &fracs).unwrap();
    let (tr, va) = (lc.mean_train_scores(), lc.mean_val_scores());
    (lc.train_sizes.iter().zip(tr).map(|(&s, v)| (s as f64, v)).collect(),
     lc.train_sizes.iter().zip(va).map(|(&s, v)| (s as f64, v)).collect())
};

evcxr_figure((580, 420), |root| {
    root.fill(&WHITE)?;
    let x0 = tr_curve.first().unwrap().0;
    let x1 = tr_curve.last().unwrap().0;
    let mut chart = ChartBuilder::on(&root)
        .caption("learning curve (logistic regression, 5-fold CV)", ("sans-serif", 16))
        .margin(10).x_label_area_size(36).y_label_area_size(44)
        .build_cartesian_2d(x0..x1, 0.80f64..1.01f64)?;
    chart.configure_mesh().x_desc("training samples").y_desc("accuracy").draw()?;
    chart.draw_series(LineSeries::new(tr_curve.clone(), BLUE.stroke_width(2)))?
        .label("train").legend(|(x, y)| PathElement::new(vec![(x, y), (x + 18, y)], BLUE));
    chart.draw_series(tr_curve.iter().map(|p| Circle::new(*p, 3, BLUE.filled())))?;
    chart.draw_series(LineSeries::new(va_curve.clone(), RED.stroke_width(2)))?
        .label("validation").legend(|(x, y)| PathElement::new(vec![(x, y), (x + 18, y)], RED));
    chart.draw_series(va_curve.iter().map(|p| Circle::new(*p, 3, RED.filled())))?;
    chart.configure_series_labels().position(SeriesLabelPosition::LowerRight)
        .background_style(WHITE.mix(0.85)).border_style(BLACK).draw()?;
    Ok(())
})

The train curve starts near-perfect (easy to fit a handful of points) and
settles as data grows; the validation curve climbs toward it. A persistent gap
would point to variance — the concrete fix is the **regularization** from the
Regression chapters (Ridge/Lasso), which trades a little training fit for better
generalization.

## Validation curve — how complex should the model be?

Now hold the data fixed and vary a **hyperparameter** instead. A decision tree's
`max_depth` is the classic knob: too shallow underfits, too deep overfits by
memorising the training set. The train/validation gap widening as depth grows is
overfitting made visible.

In [ ]:
use model_selection_rs::evaluate::validation_curve;
use smartcore::tree::decision_tree_classifier::{DecisionTreeClassifier, DecisionTreeClassifierParameters};

// `validation_curve` sweeps one hyperparameter, CV-scoring each value. The
// fit-closure now takes the parameter (max_depth) as its first argument.
let (tr_curve, va_curve): (Vec<(f64, f64)>, Vec<(f64, f64)>) = {
    let fit_depth = |depth: &u16, x: &Array2<f64>, y: &Array1<f64>| {
        let yi: Vec<i32> = y.iter().map(|&v| v as i32).collect();
        let params = DecisionTreeClassifierParameters::default().with_max_depth(*depth);
        let model = DecisionTreeClassifier::fit(&to_dm(x), &yi, params).unwrap();
        move |xq: &Array2<f64>| Array1::from(
            model.predict(&to_dm(xq)).unwrap().iter().map(|&v| v as f64).collect::<Vec<f64>>())
    };
    let kf = MsKFold::new(5).unwrap().with_shuffle(42);
    let depths: Vec<u16> = (1..=12).collect();
    let vc = validation_curve(&kf, &xa, &ya, fit_depth, &Accuracy, &depths).unwrap();
    let (tr, va) = (vc.mean_train_scores(), vc.mean_val_scores());
    (vc.param_values.iter().zip(tr).map(|(&d, v)| (d as f64, v)).collect(),
     vc.param_values.iter().zip(va).map(|(&d, v)| (d as f64, v)).collect())
};

evcxr_figure((580, 420), |root| {
    root.fill(&WHITE)?;
    let mut chart = ChartBuilder::on(&root)
        .caption("validation curve (tree max_depth, 5-fold CV)", ("sans-serif", 16))
        .margin(10).x_label_area_size(36).y_label_area_size(44)
        .build_cartesian_2d(1f64..12f64, 0.80f64..1.01f64)?;
    chart.configure_mesh().x_desc("max_depth").y_desc("accuracy").draw()?;
    chart.draw_series(LineSeries::new(tr_curve.clone(), BLUE.stroke_width(2)))?
        .label("train").legend(|(x, y)| PathElement::new(vec![(x, y), (x + 18, y)], BLUE));
    chart.draw_series(tr_curve.iter().map(|p| Circle::new(*p, 3, BLUE.filled())))?;
    chart.draw_series(LineSeries::new(va_curve.clone(), RED.stroke_width(2)))?
        .label("validation").legend(|(x, y)| PathElement::new(vec![(x, y), (x + 18, y)], RED));
    chart.draw_series(va_curve.iter().map(|p| Circle::new(*p, 3, RED.filled())))?;
    chart.configure_series_labels().position(SeriesLabelPosition::LowerRight)
        .background_style(WHITE.mix(0.85)).border_style(BLACK).draw()?;
    Ok(())
})

## Learning vs validation curves — don't conflate them

| Curve | X-axis varies | Answers |
| --- | --- | --- |
| **Learning curve** | training-set **size** | Would more data help? Bias vs. variance at fixed complexity. |
| **Validation curve** | one **hyperparameter** | What complexity generalizes best? Where does overfitting start? |

Both use the train-vs-validation gap as the diagnostic — but tuning the *knob*
(validation curve) is the job of the [Optimization](../05b-optimization/hyperparameter-search.ipynb)
chapter's hyperparameter search, done systematically instead of by eye.

Next: back to modelling, now equipped to evaluate and diagnose every model you
build.